# ccdp — train the VMMRdb make/model/year identifier on Kaggle

Continue-trains the 196-class Stanford identifier on the **full VMMRdb (~9,170 classes)** — which flows
straight into Variant D / multi-car costing (more cars named → better make/segment cost).

### Before you run — notebook settings (right sidebar)
1. **Accelerator** → GPU (T4 ×2 or P100).
2. **Internet** → On (to clone the repo + fetch the base weights).
3. **Add Input** → search **`vmmrdb-dataset`** (prabashwara) and add it — mounts read-only under `/kaggle/input/`.

Full dataset (~9,170 classes, ~291k images), ~12 epochs ≈ **6–10 h** on a T4 — fits a 12 h session.

### ⚠️ Don't run in interactive mode — use Save & Run All
Kaggle's interactive runner throttles/pauses when the browser tab loses focus, so a "leave it running"
tab will hang. Use **Save Version → Save & Run All (Commit)** (top-right). The notebook runs server-side,
you can close the browser, and Kaggle emails you when it finishes. Run history + outputs are kept as a
notebook version.

### Crash insurance — GDrive checkpoint sync
We also rsync `checkpoints/identifier/run_*/` to your Google Drive after every epoch via `rclone`. If the
kernel dies at epoch 10/12, the next Save&Run-All re-pulls the checkpoints and resumes with
`ccdp train identifier-continue --resume … --resume-run-dir …` (see the resume cell below). Set up
once: add a **Kaggle Secret** named `RCLONE_CONF` containing your rclone config (`gdrive:` remote).
Skip the sync cells if you don't want this — you'll just lose progress on a crash.


## 1. GPU check


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'enable GPU in settings')

## 2. Clone repo + install

**⚠️ Turn Internet ON first** (Settings → Internet) — Kaggle blocks network by default, so without it
`git clone` fails with *Could not resolve host*. Enabling it needs a phone-verified Kaggle account (free).


In [ ]:
%cd /kaggle/working
# NOTE: requires Internet = ON (Settings panel). Kaggle needs a phone-verified account.
# Private repo? use a token:  https://<TOKEN>@github.com/theDocWho/...
!rm -rf car-crash-fix-amount-predictor
!git clone --depth 1 https://github.com/theDocWho/car-crash-fix-amount-predictor.git
%cd /kaggle/working/car-crash-fix-amount-predictor
!pip -q install -e '.[ml]'

## 3. SSL cert (for fetching the base weights)


In [ ]:
import os, certifi
os.environ['SSL_CERT_FILE']=certifi.where()
os.environ['REQUESTS_CA_BUNDLE']=certifi.where()
print('ok')

## 4. Point the loader at the mounted VMMRdb dataset (no download)

Symlinks the Kaggle input into the path the loader expects; auto-detects the dataset folder by name.


In [ ]:
import os, glob
cands = [d for d in glob.glob('/kaggle/input/*') if 'vmmr' in d.lower()] or glob.glob('/kaggle/input/*')
assert cands, 'Add the vmmrdb-dataset input via Add Data (right sidebar).'
src = cands[0]; print('VMMRdb input:', src)
os.makedirs('data/raw', exist_ok=True)
os.system(f"ln -sfn '{src}' data/raw/vmmrdb-dataset/prabashwara/vmmrdb-dataset")
# sanity: count class folders (dirs with images)
from ccdp.data import vmmrdb
counts = vmmrdb._class_dir_counts('data/raw/vmmrdb-dataset')
print(f'class folders found: {len(counts)}  total images: {sum(counts.values())}')

## 5. Fetch the base identifier (196-class) to warm-start from


In [ ]:
!ccdp costing init || true
!mkdir -p checkpoints/production
![ -f checkpoints/production/identifier.pt] || curl -fL --retry 3 -o checkpoints/production/identifier.pt \
   https://github.com/theDocWho/car-crash-fix-amount-predictor/releases/download/v0.2.0/identifier.pt

## 5b. (Optional) GDrive sync — crash insurance

Configures `rclone` from a Kaggle Secret called **`RCLONE_CONF`** (Add-ons → Secrets). Paste the contents
of your local `~/.config/rclone/rclone.conf` (with a remote named `gdrive:`) into the secret. We then:

1. Restore any previous run's `checkpoints/identifier/run_*/` from GDrive (so we can resume on next launch).
2. Start a background thread that uploads new checkpoints every 60s — at worst we lose one in-progress epoch.

Skip the next two cells if you don't want sync. The trainer still saves per-epoch locally; you just won't
survive a kernel crash.

In [ ]:
## 6. Continue-train on the full VMMRdb (~9,170 classes)

Warm-starts the 196-class ResNet-50, swaps the head to ~9,170 classes, two-stage fine-tunes
(2 frozen + 10 unfrozen = **12 epochs** total). `--top-n 0` = full dataset; add `--epochs-stage2 6`
for a shorter run.

**Auto-resume:** if the GDrive cell above restored a `run_*identifier_vmmrdb/last.pt`, the next
cell passes `--resume <last.pt> --resume-run-dir <run_dir>` so training picks up at the next epoch
into the same run directory (same id, same metrics file). Otherwise it starts a fresh run.


In [ ]:
# Decide fresh vs resume based on what GDrive restored. Builds the right CLI flags either way.
import glob, os, shlex, subprocess, pathlib

prior_runs = sorted(glob.glob("checkpoints/identifier/run_*identifier_vmmrdb"))
resume_flags = ""
if prior_runs:
    run_dir = prior_runs[-1]
    last_pt = pathlib.Path(run_dir) / "last.pt"
    if last_pt.exists():
        resume_flags = f"--resume {shlex.quote(str(last_pt))} --resume-run-dir {shlex.quote(run_dir)}"
        print(f"[resume] picking up from {last_pt}")
    else:
        print(f"[resume] found {run_dir} but no last.pt — starting fresh")
else:
    print("[resume] no prior runs — starting fresh")

cmd = (
    "ccdp train identifier-continue "
    "--dataset vmmrdb --top-n 0 --batch-size 64 --num-workers 4 "
    f"{resume_flags}"
)
print(f"$ {cmd}")
os.system(cmd)


## 6. Continue-train on the full VMMRdb (~9,170 classes)

Warm-starts the 196-class ResNet-50, swaps the head to ~9,170 classes, two-stage fine-tunes. `--top-n 0` = full dataset; add `--epochs-stage2 6`
for a shorter run.


In [ ]:
!ccdp train identifier-continue --dataset vmmrdb --top-n 0 --batch-size 64 --num-workers 4

## 7. Promote + export the trained identifier


## 9. If the run crashed — resume flow

The Save & Run All session keeps the browser tab irrelevant, but kernels can still die
(OOT at 11h59m, OOM, preemption). Here's the recovery loop:

1. Open the notebook fresh.
2. **Don't change anything.** Run all cells top-to-bottom (or hit Save & Run All again).
3. The GDrive cell (5b) downloads the most recent `run_*identifier_vmmrdb/` back into
   `checkpoints/identifier/`.
4. The training cell auto-detects `last.pt` and adds `--resume <path> --resume-run-dir <dir>`.
5. `ccdp train identifier-continue` reads the saved state (model weights, epoch counter,
   stage, `best_val`) and starts the loop at `epoch + 1`. Stage-2 unfreeze + LR drop are
   re-applied if the saved stage was 2.
6. Training writes new `epoch_NNN.pt` into the **same** run dir, so `best.pt` / `last.pt`
   symlinks and `metrics.json` keep accumulating instead of forking a new run id.

So a 9/12 crash means epoch 10 starts immediately on the next launch — no recompute of
epochs 1–9. The `config.yaml` is rewritten each launch but the in-flight `resume_from`
field captures the lineage.

**Sanity check before relying on this:** the first time you set up `RCLONE_CONF`, run
just the GDrive cell once; you should see `[gdrive] rclone configured OK`. If the secret
is missing, the cell prints a skip message and you fall back to single-shot training.


In [ ]:
!ccdp registry list --variant identifier | tail -5
# promote the run you just trained (replace <run_id> with the id above):
# !ccdp registry promote <run_id> identifier
import shutil, glob, os
src = sorted(glob.glob('checkpoints/identifier/run_*identifier_vmmrdb/best.pt'))
if src:
    shutil.copy(os.path.realpath(src[-1]), '/kaggle/working/identifier.pt')
    print('exported -> /kaggle/working/identifier.pt  (download from the Output tab)')
else:
    print('no best.pt yet — check the training output above')

## 8. Deploy

Download `identifier.pt` from the **Output** tab (or save the notebook output), then attach it to the GitHub release
the app pulls from:
```bash
gh release upload v0.2.0 identifier.pt --clobber
```
On the next Space boot the new identifier is fetched and recognises the full VMMRdb make/model set — Variant D and multi-car mode
benefit automatically (more cars named → better make/segment cost). The model self-describes its classes (names
embedded in the checkpoint), so no extra mapping file is needed.
